# **Upload Data**

In [4]:
import pandas as pd
import numpy as np

github_url = "https://raw.githubusercontent.com/DuaA-A/Enterprise-E-Commerce-Predictive-Analytics/main/Medallion%20Architecture%20/Broze%20Layer/raw_data.csv"

df = pd.read_csv(github_url)

print("Shape:", df.shape)
df.head()

Shape: (13324, 37)


,Date,Title,Account Title,Market Place,SKU,FNSKU,ASIN,Parent ASIN,Is Parent,Internal Name,...,Impressions,Clicks,PPC Orders,PPC Sales,PPC Cost,PPC Conv,OOE,Net Profit,Net Margin,Net ROI
0,2020-10-01,High Absorption Magnesium Citrate Supplements ...,[ UK ] Aava,UK,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,NaN,0.0,NaN,...,3423.0,13.0,2.0,29.36,-13.06,0.1538,0.0,56.28,0.1852,0.3142
1,2020-10-02,High Absorption Magnesium Citrate Supplements ...,[ UK ] Aava,UK,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,NaN,0.0,NaN,...,2786.0,7.0,0.0,0.00,-7.33,0.0000,0.0,71.73,0.2034,0.3439
2,2020-10-03,High Absorption Magnesium Citrate Supplements ...,[ UK ] Aava,UK,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,NaN,0.0,NaN,...,2927.0,7.0,1.0,14.69,-6.15,0.1429,0.0,61.37,0.1386,0.2416
3,2020-10-04,High Absorption Magnesium Citrate Supplements ...,[ UK ] Aava,UK,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,NaN,0.0,NaN,...,3584.0,16.0,2.0,44.07,-16.51,0.1250,0.0,39.49,0.1573,0.2684
4,2020-10-05,High Absorption Magnesium Citrate Supplements ...,[ UK ] Aava,UK,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,NaN,0.0,NaN,...,4329.0,7.0,0.0,0.00,-4.47,0.0000,0.0,59.86,0.2116,0.3588


# **Data Exploration**

In [5]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13324 entries, 0 to 13323
Data columns (total 37 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Date              13323 non-null  object 
 1   Title             13233 non-null  object 
 2   Account Title     13323 non-null  object 
 3   Market Place      13323 non-null  object 
 4   SKU               13323 non-null  object 
 5   FNSKU             13323 non-null  object 
 6   ASIN              13284 non-null  object 
 7   Parent ASIN       2130 non-null   object 
 8   Is Parent         13323 non-null  float64
 9   Internal Name     0 non-null      float64
 10  Brand             13233 non-null  object 
 11  Product Group     0 non-null      float64
 12  Taxes             13323 non-null  float64
 13  Orders            13323 non-null  float64
 14  Units             13323 non-null  float64
 15  Refunded          13323 non-null  float64
 16  Refund %          13323 non-null  float6

In [6]:

print(df.isnull().sum())

Date                    1
Title                  91
Account Title           1
Market Place            1
SKU                     1
FNSKU                   1
ASIN                   40
Parent ASIN         11194
Is Parent               1
Internal Name       13324
Brand                  91
Product Group       13324
Taxes                   1
Orders                  1
Units                   1
Refunded                1
Refund %                1
Unit Session %          1
Promo Units             1
Organic Units           1
Per Unit Revenue     1328
Revenue                 0
COGS                    1
FBA Fees                1
Promo Amount            1
Sessions                1
Page Views              1
Impressions             1
Clicks                  1
PPC Orders              1
PPC Sales               1
PPC Cost                1
PPC Conv                1
OOE                     1
Net Profit              1
Net Margin              1
Net ROI                 1
dtype: int64


# **Data Cleaning**

## Remove Extra Spaces From Column Names

In [7]:
df.columns = df.columns.str.strip()

## Remove Duplicate Rows


In [8]:
duplicates = df.duplicated().sum()

df.drop_duplicates(inplace=True)

## Drop Completely Empty Columns

In [9]:
empty_cols = df.columns[df.isnull().all()]

df.drop(columns=empty_cols, inplace=True)

## Convert Date Column

In [10]:
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(
        df['Date'],
        errors='coerce'
    )

## Clean Text Columns

In [11]:
text_cols = df.select_dtypes(include='object').columns

for col in text_cols:

    # remove spaces
    df[col] = df[col].astype(str).str.strip()

    # replace empty strings with NaN
    df[col] = df[col].replace('', np.nan)


## Fill Parent ASIN

In [12]:
df['Parent ASIN'] = df['Parent ASIN'].replace('nan', np.nan)
df['Parent ASIN'] = df['Parent ASIN'].fillna('No Parent')

## Fill Missing Values In Text Columns

In [13]:
text_cols = df.select_dtypes(include='object').columns

for col in text_cols:

    if df[col].isnull().sum() > 0:

        mode_value = df[col].mode()

        if len(mode_value) > 0:
            df[col].fillna(
                mode_value[0],
                inplace=True
            )
        else:
            df[col].fillna(
                'Unknown',
                inplace=True
            )

## Fill Per Unit Revenue

In [14]:
if (
    'Per Unit Revenue' in df.columns and
    'Revenue' in df.columns and
    'Units' in df.columns
):

    calculated_value = (
        df['Revenue'] /
        df['Units'].replace(0, np.nan)
    )

    df['Per Unit Revenue'] = (
        df['Per Unit Revenue']
        .fillna(calculated_value)
    )

## Fill Missing Numeric Values

In [15]:
numeric_cols = df.select_dtypes(
    include=['int64', 'float64']
).columns

for col in numeric_cols:

    if df[col].isnull().sum() > 0:

        median_value = df[col].median()

        df[col].fillna(
            median_value,
            inplace=True
        )

/tmp/ipykernel_4293/523771868.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(


## Remove Impossible Negative Values

In [16]:
important_positive_cols = [

    'Orders',
    'Units',
    'Revenue',
    'Sessions',
    'Page Views',
    'Impressions',
    'Clicks',
    'PPC Orders',
    'PPC Sales'

]

for col in important_positive_cols:

    if col in df.columns:

        df = df[df[col] >= 0]

## Standardize Brand Names

In [17]:
if 'Brand' in df.columns:

    df['Brand'] = (
        df['Brand']
        .astype(str)
        .str.title()
    )

## Detect Outliers

In [18]:
if 'Revenue' in df.columns:

    Q1 = df['Revenue'].quantile(0.25)

    Q3 = df['Revenue'].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - (1.5 * IQR)

    upper_bound = Q3 + (1.5 * IQR)

    outliers = df[
        (df['Revenue'] < lower_bound) |
        (df['Revenue'] > upper_bound)
    ]

print("\nDataset Shape:")
print(df.shape)

print("\nMissing Values:")
print(df.isnull().sum().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())


Dataset Shape:
(13324, 35)

Missing Values:
1

Duplicate Rows:
0


## Replace Nan Values

In [19]:
import numpy as np

for col in df.select_dtypes(include='object').columns:

    df[col] = df[col].replace(
        ['nan', 'Nan', 'NaN', 'None'],
        np.nan
    )

for col in df.select_dtypes(include='object').columns:

    if df[col].isnull().sum() > 0:

        df[col] = df[col].fillna(
            df[col].mode()[0]
        )

## Replace Nan Value in Date

In [20]:
df = df.dropna(subset=['Date'])


# **Test**

In [21]:
print(df.isnull().sum())

Date                0
Title               0
Account Title       0
Market Place        0
SKU                 0
FNSKU               0
ASIN                0
Parent ASIN         0
Is Parent           0
Brand               0
Taxes               0
Orders              0
Units               0
Refunded            0
Refund %            0
Unit Session %      0
Promo Units         0
Organic Units       0
Per Unit Revenue    0
Revenue             0
COGS                0
FBA Fees            0
Promo Amount        0
Sessions            0
Page Views          0
Impressions         0
Clicks              0
PPC Orders          0
PPC Sales           0
PPC Cost            0
PPC Conv            0
OOE                 0
Net Profit          0
Net Margin          0
Net ROI             0
dtype: int64


In [22]:
print("\nDataset Shape:")
print(df.shape)

print("\nMissing Values:")
print(df.isnull().sum().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())


Dataset Shape:
(13323, 35)

Missing Values:
0

Duplicate Rows:
0


In [23]:
df

,Date,Title,Account Title,Market Place,SKU,FNSKU,ASIN,Parent ASIN,Is Parent,Brand,...,Impressions,Clicks,PPC Orders,PPC Sales,PPC Cost,PPC Conv,OOE,Net Profit,Net Margin,Net ROI
0,2020-10-01,High Absorption Magnesium Citrate Supplements ...,[ UK ] Aava,UK,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,No Parent,0.0,Aavalabs,...,3423.0,13.0,2.0,29.36,-13.06,0.1538,0.0,56.28,0.1852,0.3142
1,2020-10-02,High Absorption Magnesium Citrate Supplements ...,[ UK ] Aava,UK,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,No Parent,0.0,Aavalabs,...,2786.0,7.0,0.0,0.00,-7.33,0.0000,0.0,71.73,0.2034,0.3439
2,2020-10-03,High Absorption Magnesium Citrate Supplements ...,[ UK ] Aava,UK,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,No Parent,0.0,Aavalabs,...,2927.0,7.0,1.0,14.69,-6.15,0.1429,0.0,61.37,0.1386,0.2416
3,2020-10-04,High Absorption Magnesium Citrate Supplements ...,[ UK ] Aava,UK,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,No Parent,0.0,Aavalabs,...,3584.0,16.0,2.0,44.07,-16.51,0.1250,0.0,39.49,0.1573,0.2684
4,2020-10-05,High Absorption Magnesium Citrate Supplements ...,[ UK ] Aava,UK,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,No Parent,0.0,Aavalabs,...,4329.0,7.0,0.0,0.00,-4.47,0.0000,0.0,59.86,0.2116,0.3588
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13318,2021-03-27,Gélules biodisponibles du complexe de curcumin...,[ FR ] Aava,FR,FR_Aava_Turmeric_1400mg_180_01,X000UL0561,B07CXYM3H5,No Parent,0.0,Aavalabs,...,14.0,0.0,0.0,0.00,0.00,0.0000,0.0,7.37,0.1849,0.2626
13319,2021-03-28,Gélules biodisponibles du complexe de curcumin...,[ FR ] Aava,FR,FR_Aava_Turmeric_1400mg_180_01,X000UL0561,B07CXYM3H5,No Parent,0.0,Aavalabs,...,37.0,0.0,0.0,0.00,0.00,0.0000,0.0,13.96,0.1867,0.2493
13320,2021-03-29,Gélules biodisponibles du complexe de curcumin...,[ FR ] Aava,FR,FR_Aava_Turmeric_1400mg_180_01,X000UL0561,B07CXYM3H5,No Parent,0.0,Aavalabs,...,16.0,0.0,0.0,0.00,0.00,0.0000,0.0,38.34,0.2800,0.4193
13321,2021-03-30,Gélules biodisponibles du complexe de curcumin...,[ FR ] Aava,FR,FR_Aava_Turmeric_1400mg_180_01,X000UL0561,B07CXYM3H5,No Parent,0.0,Aavalabs,...,57.0,1.0,2.0,36.94,-0.71,2.0000,0.0,48.72,0.2693,0.4193


# **Download Cleaned Data**

In [24]:
import pandas as pd

# Converted Date
df['Date'] = pd.to_datetime(df['Date']).dt.date

percent_cols = [
    'Refund %',
    'Unit Session %',
    'Net Margin',
    'Net ROI'
]

with pd.ExcelWriter(
    'cleaned_amazon_sales.xlsx',
    engine='openpyxl',
    datetime_format='yyyy-mm-dd'
) as writer:

    df.to_excel(
        writer,
        index=False,
        sheet_name='Sales'
    )

    ws = writer.sheets['Sales']

    # Percentage
    for col_num, col_name in enumerate(df.columns, start=1):

        if col_name in percent_cols:

            for row in range(2, len(df) + 2):

                ws.cell(
                    row=row,
                    column=col_num
                ).number_format = '0.00%'

print("Excel file saved successfully!")

Excel file saved successfully!


In [25]:
from google.colab import files

files.download('cleaned_amazon_sales.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>